In [1]:
!pip install -U google-genai pandas tqdm python-dotenv

In [2]:
from __future__ import annotations

import os
import re
import json
import uuid
import hashlib
import datetime as dt
from pathlib import Path
from typing import Any, Optional

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

from google import genai
from google.genai import types

load_dotenv()

# You can either set GEMINI_API_KEY / GOOGLE_API_KEY in .env,
# or directly paste the key here.
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# Example if you prefer direct key:
GEMINI_API_KEY = ""

assert GEMINI_API_KEY, "Missing GEMINI_API_KEY or GOOGLE_API_KEY."

client = genai.Client(api_key=GEMINI_API_KEY)

# -----------------------------
# Experiment configuration
# -----------------------------
PROVIDER = "gemini"
MODEL_NAME = "gemini-2.5-pro"
TASK_SET_ID = "taskset_full_all_prompts_gemini25pro"

TEMPERATURE = 1.0

# Gemini 2.5 uses thinking_budget, not thinking_level.
# Use the lowest safe Pro budget to reduce latency/cost while avoiding invalid 0-budget behavior.
THINKING_LEVEL = None
THINKING_BUDGET = 128

GEMINI_BATCH_FIELD_STYLE = "snake"
GEMINI_UPLOAD_MIME_TYPE = "jsonl"
INCLUDE_THINKING_CONFIG_IN_BATCH = True

N_BASE_AGENTS = 150
N_DYADS = 75
N_TRIADS = 50

# Larger than visible answer needs so thinking does not eat the output budget.
MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 512,
    "aut": 768,
    "story": 2048,
}

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / PROVIDER
    / f"model_{MODEL_NAME}"
    / TASK_SET_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "preflight_plans": DATA_ROOT / "00_tiny_file_preflight" / "plans",
    "preflight_batch_inputs": DATA_ROOT / "00_tiny_file_preflight" / "batch_inputs",
    "preflight_uploaded_files": DATA_ROOT / "00_tiny_file_preflight" / "uploaded_files",
    "preflight_manifests": DATA_ROOT / "00_tiny_file_preflight" / "manifests",
    "preflight_raw_outputs": DATA_ROOT / "00_tiny_file_preflight" / "raw_outputs",
    "preflight_parsed": DATA_ROOT / "00_tiny_file_preflight" / "parsed",
    "round1_plans": DATA_ROOT / "01_round1" / "plans",
    "round1_batch_inputs": DATA_ROOT / "01_round1" / "batch_inputs",
    "round1_uploaded_files": DATA_ROOT / "01_round1" / "uploaded_files",
    "round1_manifests": DATA_ROOT / "01_round1" / "manifests",
    "round1_raw_outputs": DATA_ROOT / "01_round1" / "raw_outputs",
    "round1_parsed": DATA_ROOT / "01_round1" / "parsed",
    "round2_plans": DATA_ROOT / "02_round2" / "plans",
    "round2_batch_inputs": DATA_ROOT / "02_round2" / "batch_inputs",
    "round2_uploaded_files": DATA_ROOT / "02_round2" / "uploaded_files",
    "round2_manifests": DATA_ROOT / "02_round2" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "02_round2" / "raw_outputs",
    "round2_parsed": DATA_ROOT / "02_round2" / "parsed",
    "compiled": DATA_ROOT / "03_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

/Users/raiyanabdulbaten/miniforge3/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/raiyanabdulbaten/miniforge3/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


Run directory:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c


In [3]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None

    text = str(text).strip()

    text = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()

    return text


def to_jsonable(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    try:
        return json.loads(json.dumps(obj, default=str))
    except Exception:
        return str(obj)


def enum_name(x: Any) -> str:
    if x is None:
        return ""
    if hasattr(x, "name"):
        return x.name
    return str(x)

In [4]:
run_config = {
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "task_set_id": TASK_SET_ID,
    "temperature": TEMPERATURE,
    "thinking_level": THINKING_LEVEL,
    "thinking_budget": THINKING_BUDGET,
    "gemini_batch_field_style": GEMINI_BATCH_FIELD_STYLE,
    "gemini_upload_mime_type": GEMINI_UPLOAD_MIME_TYPE,
    "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
    "n_base_agents": N_BASE_AGENTS,
    "n_dyads": N_DYADS,
    "n_triads": N_TRIADS,
    "max_output_tokens_by_family": MAX_OUTPUT_TOKENS_BY_FAMILY,
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{TASK_SET_ID}__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/00_metadata/experiment_config__taskset_full_all_prompts_gemini25pro__20260518_122755__a251551c.json')

In [5]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_smartphone",
        "task_family": "slogan",
        "task_label": "Smartphone slogan",
        "task_prompt_key": "smartphone",
    },
    {
        "task_id": "slogan_soda",
        "task_family": "slogan",
        "task_label": "Soda slogan",
        "task_prompt_key": "soda",
    },
    {
        "task_id": "slogan_blood_donation",
        "task_family": "slogan",
        "task_label": "Blood donation slogan",
        "task_prompt_key": "blood_donation",
    },
    {
        "task_id": "aut_shoe",
        "task_family": "aut",
        "task_label": "AUT shoe",
        "task_prompt_key": "shoe",
        "object": "shoe",
        "common_use": "used as footwear",
    },
    {
        "task_id": "aut_button",
        "task_family": "aut",
        "task_label": "AUT button",
        "task_prompt_key": "button",
        "object": "button",
        "common_use": "used to fasten things",
    },
    {
        "task_id": "aut_key",
        "task_family": "aut",
        "task_label": "AUT key",
        "task_prompt_key": "key",
        "object": "key",
        "common_use": "used to open a lock",
    },
    {
        "task_id": "aut_wooden_pencil",
        "task_family": "aut",
        "task_label": "AUT wooden pencil",
        "task_prompt_key": "wooden_pencil",
        "object": "wooden pencil",
        "common_use": "used for writing",
    },
    {
        "task_id": "aut_automobile_tire",
        "task_family": "aut",
        "task_label": "AUT automobile tire",
        "task_prompt_key": "automobile_tire",
        "object": "automobile tire",
        "common_use": "used on the wheel of an automobile",
    },
    {
        "task_id": "story_jungle",
        "task_family": "story",
        "task_label": "Jungle adventure story",
        "task_prompt_key": "jungle",
    },
    {
        "task_id": "story_parachute",
        "task_family": "story",
        "task_label": "Parachute story",
        "task_prompt_key": "parachute",
    },
    {
        "task_id": "story_horror",
        "task_family": "story",
        "task_label": "Horror story",
        "task_prompt_key": "horror",
    },
    {
        "task_id": "story_life_last_seconds",
        "task_family": "story",
        "task_label": "Life and last seconds story",
        "task_prompt_key": "life_last_seconds",
    },
]

STRATEGIES = ["vanilla", "diverge"]
CONDITIONS = ["base", "dyad", "triad"]

tasks_df = pd.DataFrame(TASK_SETTINGS)

display(tasks_df)

print("Number of task settings:", len(TASK_SETTINGS))

,task_id,task_family,task_label,task_prompt_key,object,common_use
0,slogan_smartphone,slogan,Smartphone slogan,smartphone,NaN,NaN
1,slogan_soda,slogan,Soda slogan,soda,NaN,NaN
2,slogan_blood_donation,slogan,Blood donation slogan,blood_donation,NaN,NaN
3,aut_shoe,aut,AUT shoe,shoe,shoe,used as footwear
4,aut_button,aut,AUT button,button,button,used to fasten things
5,aut_key,aut,AUT key,key,key,used to open a lock
6,aut_wooden_pencil,aut,AUT wooden pencil,wooden_pencil,wooden pencil,used for writing
7,aut_automobile_tire,aut,AUT automobile tire,automobile_tire,automobile tire,used on the wheel of an automobile
8,story_jungle,story,Jungle adventure story,jungle,NaN,NaN
9,story_parachute,story,Parachute story,parachute,NaN,NaN


Number of task settings: 12


In [6]:
SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_smartphone":
        return (
            "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\n"
            "Generate exactly one marketing slogan for this brand-new smartphone.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the smartphone.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_soda":
        return (
            "You are part of the marketing team at a beverage company preparing to launch a new soda.\n\n"
            "Generate exactly one marketing slogan for this brand-new soda.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the soda.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_blood_donation":
        return (
            "You are part of the communications team at a nonprofit organization preparing a campaign "
            "to encourage blood donation.\n\n"
            "Generate exactly one campaign slogan for this blood donation campaign.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the campaign.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {
        "aut_shoe",
        "aut_button",
        "aut_key",
        "aut_wooden_pencil",
        "aut_automobile_tire",
    }:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_jungle":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story about an adventure in the jungle.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_parachute":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story based on this prompt:\n"
            "The parachute isn’t opening up.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_horror":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one short horror story designed to chill the bones.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_life_last_seconds":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story in 8 sentences. The first sentence must describe 100 years of a character's life. "
            "The next 7 sentences must describe the last 10 seconds of that character's life.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not number or label the sentences.\n"
            "- Do not state which sentence does what.\n"
            "- Return only the story as one paragraph."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def build_round1_prompt(task: dict, strategy: str) -> str:
    return base_task_prompt(task) + "\n\n" + strategy_block(strategy)


def build_round2_context(
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    if condition == "base":
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
        )

    elif condition == "dyad":
        assert len(peer_round1_texts) == 1
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous response from another agent in the same first round:\n'
            f'"{peer_round1_texts[0]}"\n\n'
        )

    elif condition == "triad":
        assert len(peer_round1_texts) == 2
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous responses from two other agents in the same first round:\n'
            f'1. "{peer_round1_texts[0]}"\n'
            f'2. "{peer_round1_texts[1]}"\n\n'
        )

    else:
        raise ValueError(f"Unknown condition: {condition}")

    if strategy == "vanilla":
        return context + "Now generate one new response for the same task."

    if strategy == "diverge":
        return (
            context
            + "Now generate one new response for the same task. "
              "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def build_round2_prompt(
    task: dict,
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + build_round2_context(
            strategy=strategy,
            condition=condition,
            self_round1=self_round1,
            peer_round1_texts=peer_round1_texts,
        )
    )

In [7]:
def make_thinking_config_for_batch() -> dict:
    if THINKING_LEVEL is not None and THINKING_BUDGET is not None:
        raise ValueError("Use either THINKING_LEVEL or THINKING_BUDGET, not both.")

    if THINKING_LEVEL is not None:
        return {
            "thinking_level": THINKING_LEVEL,
        }

    if THINKING_BUDGET is not None:
        return {
            "thinking_budget": int(THINKING_BUDGET),
        }

    return {}


def make_gemini_generate_content_request(row: pd.Series) -> dict:
    generation_config = {
        "temperature": float(row["temperature"]),
        "max_output_tokens": int(row["max_output_tokens"]),
        "candidate_count": 1,
        "response_modalities": ["TEXT"],
    }

    thinking_config = make_thinking_config_for_batch()

    if INCLUDE_THINKING_CONFIG_IN_BATCH and thinking_config:
        generation_config["thinking_config"] = thinking_config

    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"text": row["user_prompt"]}
                ],
            }
        ],
        "system_instruction": {
            "parts": [
                {"text": row["system_instructions"]}
            ]
        },
        "generation_config": generation_config,
    }


def upload_gemini_batch_file(batch_jsonl_path: Path, display_name: str) -> dict:
    uploaded_file = client.files.upload(
        file=str(batch_jsonl_path),
        config=types.UploadFileConfig(
            display_name=display_name,
            mime_type=GEMINI_UPLOAD_MIME_TYPE,
        ),
    )
    return to_jsonable(uploaded_file)


def make_gemini_batch_jsonl_from_plan(
    plan_df: pd.DataFrame,
    round_name: str,
    batch_input_dir: Path,
    plan_dir: Path,
    stem_prefix: str,
) -> tuple[Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{stem_prefix}__{round_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = plan_dir / f"{stem}__plan.csv"
    jsonl_path = batch_input_dir / f"{stem}__batch_input.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    if plan_path.exists() or jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite existing plan or JSONL file.")

    plan_df.to_csv(plan_path, index=False)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            record = {
                "key": row["request_key"],
                "request": make_gemini_generate_content_request(row),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Wrote plan:  {plan_path}")
    print(f"Wrote JSONL: {jsonl_path}")
    print(f"Requests:    {len(plan_df):,}")

    preview = read_jsonl(jsonl_path)[0]
    print("\nFirst JSONL record preview:")
    print(json.dumps(preview, ensure_ascii=False, indent=2)[:4000])

    return jsonl_path, plan_path


def submit_gemini_batch_generic(
    batch_jsonl_path: Path,
    round_name: str,
    plan_path: Path,
    manifest_dir: Path,
    uploaded_files_dir: Path,
    display_name_prefix: str,
) -> dict:
    display_name = f"{display_name_prefix}__{TASK_SET_ID}__{round_name}__{MODEL_NAME}__{RUN_ID}"

    uploaded_dump = upload_gemini_batch_file(
        batch_jsonl_path=batch_jsonl_path,
        display_name=display_name,
    )

    uploaded_path = uploaded_files_dir / f"{round_name}__uploaded_file__{safe_slug(display_name)}.json"

    if uploaded_path.exists():
        raise FileExistsError(f"Refusing to overwrite uploaded-file manifest: {uploaded_path}")

    write_json(uploaded_path, uploaded_dump)

    uploaded_name = uploaded_dump.get("name")
    assert uploaded_name, f"Could not find uploaded file name in uploaded_dump: {uploaded_dump}"

    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded_name,
        config={
            "display_name": display_name,
        },
    )

    batch_dump = to_jsonable(batch_job)

    batch_info = {
        "run_id": RUN_ID,
        "task_set_id": TASK_SET_ID,
        "round": round_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "batch_job": batch_dump,
        "batch_name": batch_dump.get("name"),
        "state_at_submission": batch_dump.get("state"),
        "submitted_at_utc": now_iso(),
        "uploaded_file": uploaded_dump,
        "uploaded_file_manifest": str(uploaded_path),
        "batch_jsonl_path": str(batch_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = (
        manifest_dir
        / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{stable_hash(batch_info['batch_name'], 16)}.json"
    )

    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite batch manifest: {manifest_path}")

    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted Gemini batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


def check_gemini_batch(batch_name: str) -> dict:
    batch_job = client.batches.get(name=batch_name)
    info = to_jsonable(batch_job)
    print(json.dumps(info, indent=2))
    return info

In [8]:
def find_result_file_name_from_batch_dump(batch_dump: dict) -> Optional[str]:
    for parent_key in ["dest", "output", "response"]:
        parent = batch_dump.get(parent_key) or {}
        for key in ["fileName", "file_name", "name"]:
            value = parent.get(key)
            if isinstance(value, str) and value.startswith("files/"):
                return value

    def walk(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k in {"fileName", "file_name", "name"} and isinstance(v, str) and v.startswith("files/"):
                    return v
                found = walk(v)
                if found:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = walk(item)
                if found:
                    return found
        return None

    return walk(batch_dump)


def download_gemini_batch_results_generic(
    batch_name: str,
    raw_output_dir: Path,
    round_name: str,
) -> Optional[Path]:
    batch_job = client.batches.get(name=batch_name)
    batch_dump = to_jsonable(batch_job)

    state = enum_name(getattr(batch_job, "state", None)) or str(batch_dump.get("state", ""))

    if "SUCCEEDED" not in state:
        print(f"Batch has not succeeded yet. Current state: {state}")
        if batch_dump.get("error"):
            print("Batch error:")
            print(json.dumps(batch_dump.get("error"), indent=2))
        return None

    result_file_name = find_result_file_name_from_batch_dump(batch_dump)

    if not result_file_name:
        print("Could not find result file name. Batch dump preview:")
        print(json.dumps(batch_dump, indent=2)[:5000])
        raise ValueError("Could not find result file name in batch object.")

    output_path = raw_output_dir / f"{TASK_SET_ID}__{round_name}__{stable_hash(batch_name, 16)}__results.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output file: {output_path}")

    file_content = client.files.download(file=result_file_name)

    if isinstance(file_content, bytes):
        content_bytes = file_content
    elif hasattr(file_content, "read"):
        content_bytes = file_content.read()
    else:
        content_bytes = str(file_content).encode("utf-8")

    output_path.write_bytes(content_bytes)

    print(f"Downloaded results to: {output_path}")
    return output_path

In [9]:
def extract_text_from_gemini_response(response: dict) -> str:
    if not isinstance(response, dict):
        return ""

    if response.get("text"):
        return str(response["text"]).strip()

    candidates = response.get("candidates") or []
    texts = []

    for cand in candidates:
        content = cand.get("content") or {}
        parts = content.get("parts") or []

        for part in parts:
            if not isinstance(part, dict):
                continue

            if part.get("thought") is True:
                continue

            if "text" in part and part.get("text") is not None:
                texts.append(str(part.get("text", "")))

    return "\n".join(texts).strip()


def flatten_gemini_usage_metadata(response: dict) -> dict:
    usage = (
        response.get("usage_metadata")
        or response.get("usageMetadata")
        or {}
    )

    return {
        "usage_prompt_token_count": (
            usage.get("prompt_token_count")
            or usage.get("promptTokenCount")
        ),
        "usage_candidates_token_count": (
            usage.get("candidates_token_count")
            or usage.get("candidatesTokenCount")
        ),
        "usage_thoughts_token_count": (
            usage.get("thoughts_token_count")
            or usage.get("thoughtsTokenCount")
        ),
        "usage_cached_content_token_count": (
            usage.get("cached_content_token_count")
            or usage.get("cachedContentTokenCount")
        ),
        "usage_total_token_count": (
            usage.get("total_token_count")
            or usage.get("totalTokenCount")
        ),
        "usage_raw": usage,
    }


def extract_finish_reason_from_gemini_response(response: dict) -> Optional[str]:
    candidates = response.get("candidates") or []
    if not candidates:
        return None

    return (
        candidates[0].get("finish_reason")
        or candidates[0].get("finishReason")
    )


def extract_gemini_response_body_and_error(rec: dict) -> tuple[Optional[dict], Optional[Any]]:
    if not isinstance(rec, dict):
        return None, rec

    if rec.get("error") is not None:
        return None, rec.get("error")

    response = rec.get("response")

    if response is None:
        return None, rec.get("status") or rec

    if not isinstance(response, dict):
        return None, response

    if response.get("error") is not None:
        return None, response.get("error")

    if response.get("body") is not None:
        body = response.get("body")
        if isinstance(body, dict) and body.get("error") is not None:
            return None, body.get("error")
        return body, None

    if response.get("response") is not None:
        body = response.get("response")
        if isinstance(body, dict) and body.get("error") is not None:
            return None, body.get("error")
        return body, None

    if (
        response.get("candidates") is not None
        or response.get("usageMetadata") is not None
        or response.get("usage_metadata") is not None
        or response.get("text") is not None
    ):
        return response, None

    return None, response


def parse_gemini_batch_output_to_standard_df(
    batch_output_path: Path,
    plan_path: Path,
    batch_name: str,
) -> pd.DataFrame:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    batch_records = read_jsonl(batch_output_path)
    parsed_records = []

    for rec in batch_records:
        request_key = (
            rec.get("key")
            or rec.get("metadata", {}).get("key")
            or rec.get("custom_id")
        )

        plan_row = plan_by_key.get(request_key, {})

        response_body, error = extract_gemini_response_body_and_error(rec)

        if response_body is not None:
            text = clean_model_text(extract_text_from_gemini_response(response_body))
            usage_flat = flatten_gemini_usage_metadata(response_body)
            finish_reason = extract_finish_reason_from_gemini_response(response_body)

            status = "success" if text else "empty_text"

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": None,
                "finish_reason": finish_reason,
                "usage_prompt_token_count": usage_flat["usage_prompt_token_count"],
                "usage_candidates_token_count": usage_flat["usage_candidates_token_count"],
                "usage_thoughts_token_count": usage_flat["usage_thoughts_token_count"],
                "usage_cached_content_token_count": usage_flat["usage_cached_content_token_count"],
                "usage_total_token_count": usage_flat["usage_total_token_count"],
                "usage": usage_flat["usage_raw"],
                "error": None if text else "No visible final-answer text extracted from Gemini response.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_name": batch_name,
                "raw_result_type": "response",
                "raw_record": rec,
            }

        else:
            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "finish_reason": None,
                "usage_prompt_token_count": None,
                "usage_candidates_token_count": None,
                "usage_thoughts_token_count": None,
                "usage_cached_content_token_count": None,
                "usage_total_token_count": None,
                "usage": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_name": batch_name,
                "raw_result_type": "error",
                "raw_record": rec,
            }

        parsed_records.append(record)

    return pd.DataFrame(parsed_records)


def save_parsed_df(
    parsed_df: pd.DataFrame,
    parsed_dir: Path,
    round_name: str,
    batch_name: str,
    prefix: str,
) -> dict:
    batch_hash = stable_hash(batch_name, 16)

    parsed_jsonl_path = parsed_dir / f"{prefix}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{prefix}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{prefix}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    for _, row in parsed_df.iterrows():
        append_jsonl(parsed_jsonl_path, row.to_dict())

    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "task_set_id": TASK_SET_ID,
        "round": round_name,
        "batch_name": batch_name,
        "n_records": len(parsed_df),
        "n_success": int(parsed_df["status"].eq("success").sum()),
        "n_empty_text": int(parsed_df["status"].eq("empty_text").sum()),
        "n_error": int(parsed_df["status"].eq("error").sum()),
        "n_max_token_finish": int(
            parsed_df["finish_reason"]
            .astype(str)
            .str.upper()
            .str.contains("MAX|TOKEN", regex=True, na=False)
            .sum()
        ),
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{prefix}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_hash}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(summary, indent=2))
    return summary

In [10]:
def build_tiny_file_preflight_plan() -> pd.DataFrame:
    rows = []

    selected_task_ids = [
        "slogan_blood_donation",
        "story_horror",
    ]

    for task_id in selected_task_ids:
        task = next(t for t in TASK_SETTINGS if t["task_id"] == task_id)

        request_basis = {
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "task_set_id": TASK_SET_ID,
            "preflight": True,
            "preflight_case": "tiny_file_preflight_round1_diverge",
            "round": 1,
            "task_id": task["task_id"],
            "task_family": task["task_family"],
            "strategy": "diverge",
            "condition": "base",
            "group_id": "tiny_file_preflight",
            "agent_id": f"tiny_file_preflight__{task['task_id']}",
            "agent_index": 1,
        }

        request_key = "tiny_preflight__" + stable_hash(
            json.dumps(request_basis, sort_keys=True),
            24,
        )

        rows.append({
            **request_basis,
            "request_key": request_key,
            "system_instructions": SYSTEM_INSTRUCTIONS,
            "user_prompt": build_round1_prompt(task, "diverge"),
            "temperature": TEMPERATURE,
            "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
            "thinking_level": THINKING_LEVEL,
            "thinking_budget": THINKING_BUDGET,
            "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
            "created_at_utc": now_iso(),
        })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        raise RuntimeError("Duplicate tiny preflight request keys.")

    return plan_df


tiny_file_preflight_plan_df = build_tiny_file_preflight_plan()

display(tiny_file_preflight_plan_df[[
    "request_key",
    "task_id",
    "task_family",
    "strategy",
    "condition",
    "max_output_tokens",
    "thinking_level",
    "thinking_budget",
    "include_thinking_config_in_batch",
]])

,request_key,task_id,task_family,strategy,condition,max_output_tokens,thinking_level,thinking_budget,include_thinking_config_in_batch
0,tiny_preflight__c81012b6d0f97646a43a58b8,slogan_blood_donation,slogan,diverge,base,512,None,128,True
1,tiny_preflight__ca7ced5b9747d67edece4249,story_horror,story,diverge,base,2048,None,128,True


In [11]:
tiny_file_preflight_jsonl_path, tiny_file_preflight_plan_path = make_gemini_batch_jsonl_from_plan(
    plan_df=tiny_file_preflight_plan_df,
    round_name="tiny_file_preflight",
    batch_input_dir=DIRS["preflight_batch_inputs"],
    plan_dir=DIRS["preflight_plans"],
    stem_prefix=f"{TASK_SET_ID}__tiny_file_preflight",
)

print("First JSONL record:")
print(json.dumps(read_jsonl(tiny_file_preflight_jsonl_path)[0], indent=2, ensure_ascii=False)[:4000])

tiny_file_preflight_batch_info = submit_gemini_batch_generic(
    batch_jsonl_path=tiny_file_preflight_jsonl_path,
    round_name="tiny_file_preflight",
    plan_path=tiny_file_preflight_plan_path,
    manifest_dir=DIRS["preflight_manifests"],
    uploaded_files_dir=DIRS["preflight_uploaded_files"],
    display_name_prefix="deflect_creativity_tiny_file_preflight",
)

tiny_file_preflight_batch_info

Wrote plan:  ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/00_tiny_file_preflight/plans/taskset_full_all_prompts_gemini25pro__tiny_file_preflight__tiny_file_preflight__gemini__gemini-2.5-pro__20260518_122937__plan.csv
Wrote JSONL: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/00_tiny_file_preflight/batch_inputs/taskset_full_all_prompts_gemini25pro__tiny_file_preflight__tiny_file_preflight__gemini__gemini-2.5-pro__20260518_122937__batch_input.jsonl
Requests:    2

First JSONL record preview:
{
  "key": "tiny_preflight__c81012b6d0f97646a43a58b8",
  "request": {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "You are part of the communications team at a nonprofit organization preparing a campaign to encourage blood donation.\n\nGenerate exactly one campaign slogan for this blood donatio

{'run_id': '20260518_122755__a251551c',
 'task_set_id': 'taskset_full_all_prompts_gemini25pro',
 'round': 'tiny_file_preflight',
 'provider': 'gemini',
 'model': 'gemini-2.5-pro',
 'batch_job': {'name': 'batches/bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi',
  'display_name': 'deflect_creativity_tiny_file_preflight__taskset_full_all_prompts_gemini25pro__tiny_file_preflight__gemini-2.5-pro__20260518_122755__a251551c',
  'state': 'JOB_STATE_PENDING',
  'error': None,
  'create_time': '2026-05-18T16:29:38.979600Z',
  'start_time': None,
  'end_time': None,
  'update_time': '2026-05-18T16:29:38.979600Z',
  'model': 'models/gemini-2.5-pro',
  'src': None,
  'dest': None},
 'batch_name': 'batches/bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi',
 'state_at_submission': 'JOB_STATE_PENDING',
 'submitted_at_utc': '2026-05-18T16:29:39.546763+00:00',
 'uploaded_file': {'name': 'files/lta8hmlhtgs4',
  'display_name': 'deflect_creativity_tiny_file_preflight__taskset_full_all_prompts_gemini25pro__tiny_file_preflight__

In [13]:
tiny_file_preflight_status = check_gemini_batch(
    tiny_file_preflight_batch_info["batch_name"]
)

tiny_file_preflight_status

{
  "name": "batches/bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi",
  "display_name": "deflect_creativity_tiny_file_preflight__taskset_full_all_prompts_gemini25pro__tiny_file_preflight__gemini-2.5-pro__20260518_122755__a251551c",
  "state": "JOB_STATE_SUCCEEDED",
  "error": null,
  "create_time": "2026-05-18T16:29:38.979600Z",
  "start_time": null,
  "end_time": "2026-05-18T16:30:54.168902Z",
  "update_time": "2026-05-18T16:30:54.168902Z",
  "model": "models/gemini-2.5-pro",
  "src": null,
  "dest": {
    "format": null,
    "gcs_uri": null,
    "bigquery_uri": null,
    "file_name": "files/batch-bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi",
    "inlined_responses": null,
    "inlined_embed_content_responses": null
  }
}


{'name': 'batches/bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi',
 'display_name': 'deflect_creativity_tiny_file_preflight__taskset_full_all_prompts_gemini25pro__tiny_file_preflight__gemini-2.5-pro__20260518_122755__a251551c',
 'state': 'JOB_STATE_SUCCEEDED',
 'error': None,
 'create_time': '2026-05-18T16:29:38.979600Z',
 'start_time': None,
 'end_time': '2026-05-18T16:30:54.168902Z',
 'update_time': '2026-05-18T16:30:54.168902Z',
 'model': 'models/gemini-2.5-pro',
 'src': None,
 'dest': {'format': None,
  'gcs_uri': None,
  'bigquery_uri': None,
  'file_name': 'files/batch-bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi',
  'inlined_responses': None,
  'inlined_embed_content_responses': None}}

In [14]:
tiny_file_preflight_output_path = download_gemini_batch_results_generic(
    batch_name=tiny_file_preflight_batch_info["batch_name"],
    raw_output_dir=DIRS["preflight_raw_outputs"],
    round_name="tiny_file_preflight",
)

tiny_file_preflight_df = parse_gemini_batch_output_to_standard_df(
    batch_output_path=tiny_file_preflight_output_path,
    plan_path=Path(tiny_file_preflight_batch_info["plan_path"]),
    batch_name=tiny_file_preflight_batch_info["batch_name"],
)

tiny_file_preflight_parse_summary = save_parsed_df(
    parsed_df=tiny_file_preflight_df,
    parsed_dir=DIRS["preflight_parsed"],
    round_name="tiny_file_preflight",
    batch_name=tiny_file_preflight_batch_info["batch_name"],
    prefix=f"{TASK_SET_ID}__tiny_file_preflight",
)

display(tiny_file_preflight_df[[
    "task_id",
    "task_family",
    "status",
    "finish_reason",
    "usage_prompt_token_count",
    "usage_candidates_token_count",
    "usage_thoughts_token_count",
    "usage_total_token_count",
    "text",
    "error",
]])

if not tiny_file_preflight_df["status"].eq("success").all():
    for _, row in tiny_file_preflight_df.iterrows():
        print("\n" + "=" * 100)
        print(row["task_id"])
        print("=" * 100)
        print(json.dumps(row["error"], indent=2, ensure_ascii=False, default=str)[:8000])
    raise RuntimeError("Gemini 2.5 Pro tiny file preflight returned failed or empty outputs.")

if tiny_file_preflight_df["finish_reason"].astype(str).str.upper().str.contains("MAX|TOKEN", regex=True, na=False).any():
    raise RuntimeError("Gemini 2.5 Pro tiny file preflight hit a max-token finish reason.")

print("Gemini 2.5 Pro tiny preflight passed.")

Downloaded results to: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/00_tiny_file_preflight/raw_outputs/taskset_full_all_prompts_gemini25pro__tiny_file_preflight__bccedc0557815457__results.jsonl
{
  "task_set_id": "taskset_full_all_prompts_gemini25pro",
  "round": "tiny_file_preflight",
  "batch_name": "batches/bsaujruk0dg6t18c7ivcue1hwknyvt55t0hi",
  "n_records": 2,
  "n_success": 2,
  "n_empty_text": 0,
  "n_error": 0,
  "n_max_token_finish": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/00_tiny_file_preflight/parsed/taskset_full_all_prompts_gemini25pro__tiny_file_preflight__tiny_file_preflight__gemini__gemini-2.5-pro__bccedc0557815457__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/00_tiny_file_pr

,task_id,task_family,status,finish_reason,usage_prompt_token_count,usage_candidates_token_count,usage_thoughts_token_count,usage_total_token_count,text,error
0,slogan_blood_donation,slogan,success,STOP,156,8,64,228,Share a little. Save a lot.,None
1,story_horror,story,success,STOP,170,155,62,387,"The old house groaned around me, a familiar sy...",None


Gemini 2.5 Pro tiny preflight passed.


In [15]:
def build_agent_roster() -> pd.DataFrame:
    rows = []

    for i in range(1, N_BASE_AGENTS + 1):
        rows.append({
            "condition": "base",
            "group_id": f"base_{i:03d}",
            "group_size": 1,
            "agent_index": 1,
            "agent_id": f"base_{i:03d}__a1",
        })

    for g in range(1, N_DYADS + 1):
        for a in [1, 2]:
            rows.append({
                "condition": "dyad",
                "group_id": f"dyad_{g:03d}",
                "group_size": 2,
                "agent_index": a,
                "agent_id": f"dyad_{g:03d}__a{a}",
            })

    for g in range(1, N_TRIADS + 1):
        for a in [1, 2, 3]:
            rows.append({
                "condition": "triad",
                "group_id": f"triad_{g:03d}",
                "group_size": 3,
                "agent_index": a,
                "agent_id": f"triad_{g:03d}__a{a}",
            })

    return pd.DataFrame(rows)


agents_df = build_agent_roster()

display(
    agents_df.groupby("condition").agg(
        n_agents=("agent_id", "count"),
        n_groups=("group_id", "nunique"),
        group_size=("group_size", "first"),
    ).reset_index()
)

agents_df.head()

,condition,n_agents,n_groups,group_size
0,base,150,150,1
1,dyad,150,75,2
2,triad,150,50,3


,condition,group_id,group_size,agent_index,agent_id
0,base,base_001,1,1,base_001__a1
1,base,base_002,1,1,base_002__a1
2,base,base_003,1,1,base_003__a1
3,base,base_004,1,1,base_004__a1
4,base,base_005,1,1,base_005__a1


In [16]:
def build_round1_plan() -> pd.DataFrame:
    rows = []

    for task in TASK_SETTINGS:
        for strategy in STRATEGIES:
            for _, agent in agents_df.iterrows():
                user_prompt = build_round1_prompt(task, strategy)

                request_basis = {
                    "provider": PROVIDER,
                    "model": MODEL_NAME,
                    "task_set_id": TASK_SET_ID,
                    "round": 1,
                    "task_id": task["task_id"],
                    "task_family": task["task_family"],
                    "strategy": strategy,
                    "condition": agent["condition"],
                    "group_id": agent["group_id"],
                    "agent_id": agent["agent_id"],
                    "agent_index": int(agent["agent_index"]),
                }

                request_key = "r1__" + stable_hash(
                    json.dumps(request_basis, sort_keys=True),
                    24,
                )

                rows.append({
                    **request_basis,
                    "request_key": request_key,
                    "system_instructions": SYSTEM_INSTRUCTIONS,
                    "user_prompt": user_prompt,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                    "thinking_level": THINKING_LEVEL,
                    "thinking_budget": THINKING_BUDGET,
                    "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
                    "created_at_utc": now_iso(),
                })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head(20)}")

    return plan_df


round1_plan_df = build_round1_plan()

expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)

print("Expected Round 1 requests:", expected_round1)
print("Actual Round 1 requests:  ", len(round1_plan_df))

display(
    round1_plan_df
    .groupby(["task_id", "strategy", "condition"])
    .size()
    .reset_index(name="n")
)

round1_plan_df.head()

Expected Round 1 requests: 10800
Actual Round 1 requests:   10800


,task_id,strategy,condition,n
0,aut_automobile_tire,diverge,base,150
1,aut_automobile_tire,diverge,dyad,150
2,aut_automobile_tire,diverge,triad,150
3,aut_automobile_tire,vanilla,base,150
4,aut_automobile_tire,vanilla,dyad,150
...,...,...,...,...
67,story_parachute,diverge,dyad,150
68,story_parachute,diverge,triad,150
69,story_parachute,vanilla,base,150
70,story_parachute,vanilla,dyad,150


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,request_key,system_instructions,user_prompt,temperature,max_output_tokens,thinking_level,thinking_budget,include_thinking_config_in_batch,created_at_utc
0,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,slogan_smartphone,slogan,vanilla,base,base_001,base_001__a1,1,r1__9ab15aaa06a23fbc2be64a0a,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,512,None,128,True,2026-05-18T16:32:16.762168+00:00
1,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,slogan_smartphone,slogan,vanilla,base,base_002,base_002__a1,1,r1__d2bc16a455e10e64e79f6849,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,512,None,128,True,2026-05-18T16:32:16.762331+00:00
2,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,slogan_smartphone,slogan,vanilla,base,base_003,base_003__a1,1,r1__c5b9b9eeed5920557b3ed357,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,512,None,128,True,2026-05-18T16:32:16.762475+00:00
3,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,slogan_smartphone,slogan,vanilla,base,base_004,base_004__a1,1,r1__4dad4c50c602be86abc7b381,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,512,None,128,True,2026-05-18T16:32:16.762612+00:00
4,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,slogan_smartphone,slogan,vanilla,base,base_005,base_005__a1,1,r1__151139920496ec8aed11cd66,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,512,None,128,True,2026-05-18T16:32:16.762743+00:00


In [17]:
round1_jsonl_path, round1_plan_path = make_gemini_batch_jsonl_from_plan(
    plan_df=round1_plan_df,
    round_name="round1",
    batch_input_dir=DIRS["round1_batch_inputs"],
    plan_dir=DIRS["round1_plans"],
    stem_prefix=TASK_SET_ID,
)

print("Contains thinking_config?")
print("thinking_config" in json.dumps(read_jsonl(round1_jsonl_path)[0]))

round1_batch_info = submit_gemini_batch_generic(
    batch_jsonl_path=round1_jsonl_path,
    round_name="round1",
    plan_path=round1_plan_path,
    manifest_dir=DIRS["round1_manifests"],
    uploaded_files_dir=DIRS["round1_uploaded_files"],
    display_name_prefix="deflect_creativity",
)

round1_batch_info

Wrote plan:  ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/plans/taskset_full_all_prompts_gemini25pro__round1__gemini__gemini-2.5-pro__20260518_123219__plan.csv
Wrote JSONL: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/batch_inputs/taskset_full_all_prompts_gemini25pro__round1__gemini__gemini-2.5-pro__20260518_123219__batch_input.jsonl
Requests:    10,800

First JSONL record preview:
{
  "key": "r1__9ab15aaa06a23fbc2be64a0a",
  "request": {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\nGenerate exactly one marketing slogan for this brand-new smartphone.\n\nRequirements:\n- The slogan must not exceed 6 words.\n- The slogan must be written in English.\n- You may assume an

{'run_id': '20260518_122755__a251551c',
 'task_set_id': 'taskset_full_all_prompts_gemini25pro',
 'round': 'round1',
 'provider': 'gemini',
 'model': 'gemini-2.5-pro',
 'batch_job': {'name': 'batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3',
  'display_name': 'deflect_creativity__taskset_full_all_prompts_gemini25pro__round1__gemini-2.5-pro__20260518_122755__a251551c',
  'state': 'JOB_STATE_PENDING',
  'error': None,
  'create_time': '2026-05-18T16:32:22.814304Z',
  'start_time': None,
  'end_time': None,
  'update_time': '2026-05-18T16:32:22.814304Z',
  'model': 'models/gemini-2.5-pro',
  'src': None,
  'dest': None},
 'batch_name': 'batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3',
 'state_at_submission': 'JOB_STATE_PENDING',
 'submitted_at_utc': '2026-05-18T16:32:24.461781+00:00',
 'uploaded_file': {'name': 'files/i8r4yo87023i',
  'display_name': 'deflect_creativity__taskset_full_all_prompts_gemini25pro__round1__gemini-2.5-pro__20260518_122755__a251551c',
  'mime_type': 'jsonl',
  'size_byt

In [21]:
round1_status = check_gemini_batch(round1_batch_info["batch_name"])

round1_status

{
  "name": "batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3",
  "display_name": "deflect_creativity__taskset_full_all_prompts_gemini25pro__round1__gemini-2.5-pro__20260518_122755__a251551c",
  "state": "JOB_STATE_SUCCEEDED",
  "error": null,
  "create_time": "2026-05-18T16:32:22.814304Z",
  "start_time": null,
  "end_time": "2026-05-18T16:35:49.016590Z",
  "update_time": "2026-05-18T16:35:49.016590Z",
  "model": "models/gemini-2.5-pro",
  "src": null,
  "dest": {
    "format": null,
    "gcs_uri": null,
    "bigquery_uri": null,
    "file_name": "files/batch-ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3",
    "inlined_responses": null,
    "inlined_embed_content_responses": null
  }
}


{'name': 'batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3',
 'display_name': 'deflect_creativity__taskset_full_all_prompts_gemini25pro__round1__gemini-2.5-pro__20260518_122755__a251551c',
 'state': 'JOB_STATE_SUCCEEDED',
 'error': None,
 'create_time': '2026-05-18T16:32:22.814304Z',
 'start_time': None,
 'end_time': '2026-05-18T16:35:49.016590Z',
 'update_time': '2026-05-18T16:35:49.016590Z',
 'model': 'models/gemini-2.5-pro',
 'src': None,
 'dest': {'format': None,
  'gcs_uri': None,
  'bigquery_uri': None,
  'file_name': 'files/batch-ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3',
  'inlined_responses': None,
  'inlined_embed_content_responses': None}}

In [22]:
round1_output_path = download_gemini_batch_results_generic(
    batch_name=round1_batch_info["batch_name"],
    raw_output_dir=DIRS["round1_raw_outputs"],
    round_name="round1",
)

round1_output_path

Downloaded results to: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/raw_outputs/taskset_full_all_prompts_gemini25pro__round1__a46b379f5fd58c29__results.jsonl


PosixPath('ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/raw_outputs/taskset_full_all_prompts_gemini25pro__round1__a46b379f5fd58c29__results.jsonl')

In [23]:
round1_df = parse_gemini_batch_output_to_standard_df(
    batch_output_path=round1_output_path,
    plan_path=Path(round1_batch_info["plan_path"]),
    batch_name=round1_batch_info["batch_name"],
)

round1_parse_summary = save_parsed_df(
    parsed_df=round1_df,
    parsed_dir=DIRS["round1_parsed"],
    round_name="round1",
    batch_name=round1_batch_info["batch_name"],
    prefix=TASK_SET_ID,
)

print(round1_df.shape)
display(round1_df["status"].value_counts(dropna=False))
display(round1_df["finish_reason"].value_counts(dropna=False).head(20))

round1_df.head()

{
  "task_set_id": "taskset_full_all_prompts_gemini25pro",
  "round": "round1",
  "batch_name": "batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3",
  "n_records": 10800,
  "n_success": 10800,
  "n_empty_text": 0,
  "n_error": 0,
  "n_max_token_finish": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/parsed/taskset_full_all_prompts_gemini25pro__round1__gemini__gemini-2.5-pro__a46b379f5fd58c29__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/parsed/taskset_full_all_prompts_gemini25pro__round1__gemini__gemini-2.5-pro__a46b379f5fd58c29__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/01_round1/parsed/taskset_full_all_prompts_gemini25pro__round1__gemini__gemin

status
success    10800
Name: count, dtype: int64

finish_reason
STOP    10800
Name: count, dtype: int64

,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,usage_thoughts_token_count,usage_cached_content_token_count,usage_total_token_count,usage,error,batch_custom_id,batch_output_file,batch_name,raw_result_type,raw_record
0,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,story_jungle,story,diverge,dyad,dyad_051,dyad_051__a1,...,63,None,436,"{'thoughtsTokenCount': 63, 'candidatesTokenCou...",None,r1__4cee8a8658cbd08ed070980f,ai_data/deflect_creativity/gemini/model_gemini...,batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3,response,{'response': {'responseId': 'Uj8LapDrNPGeqtsPy...
1,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,story_jungle,story,diverge,dyad,dyad_051,dyad_051__a2,...,68,None,444,"{'serviceTier': 'SERVICE_TIER_STANDARD', 'thou...",None,r1__e4c9fecba5a7f8390a1dbd47,ai_data/deflect_creativity/gemini/model_gemini...,batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3,response,"{'key': 'r1__e4c9fecba5a7f8390a1dbd47', 'respo..."
2,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,story_jungle,story,diverge,dyad,dyad_052,dyad_052__a1,...,67,None,451,"{'promptTokenCount': 169, 'serviceTier': 'SERV...",None,r1__d0d6922ae44eddbc4b8f2a87,ai_data/deflect_creativity/gemini/model_gemini...,batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3,response,"{'key': 'r1__d0d6922ae44eddbc4b8f2a87', 'respo..."
3,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,story_jungle,story,diverge,dyad,dyad_052,dyad_052__a2,...,61,None,455,"{'promptTokenCount': 169, 'serviceTier': 'SERV...",None,r1__808e6c35347fe48860c05bf0,ai_data/deflect_creativity/gemini/model_gemini...,batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3,response,{'response': {'modelVersion': 'gemini-2.5-pro'...
4,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,1,story_jungle,story,diverge,dyad,dyad_053,dyad_053__a1,...,87,None,480,"{'serviceTier': 'SERVICE_TIER_STANDARD', 'thou...",None,r1__0f1625819b416fcd1220003e,ai_data/deflect_creativity/gemini/model_gemini...,batches/ajfz3t06tpvr9le0a9ygn43hq2xn7xborrv3,response,"{'key': 'r1__0f1625819b416fcd1220003e', 'respo..."


In [24]:
def validate_parsed_round(
    df: pd.DataFrame,
    expected_n: int,
    round_name: str,
    stop_on_problem: bool = True,
) -> pd.DataFrame:
    print(f"Expected {round_name} rows:", expected_n)
    print(f"Actual {round_name} rows:  ", len(df))

    print("\nStatus counts:")
    status_counts = df["status"].value_counts(dropna=False).reset_index()
    status_counts.columns = ["status", "n"]
    display(status_counts)

    print("\nFinish reason counts:")
    finish_counts = df["finish_reason"].value_counts(dropna=False).reset_index()
    finish_counts.columns = ["finish_reason", "n"]
    display(finish_counts.head(30))

    usage_cols = [
        "usage_prompt_token_count",
        "usage_candidates_token_count",
        "usage_thoughts_token_count",
        "usage_cached_content_token_count",
        "usage_total_token_count",
    ]

    existing_usage_cols = [c for c in usage_cols if c in df.columns]

    print("\nToken usage summary:")
    display(df[existing_usage_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

    row_problem_mask = (
        ~df["status"].eq("success")
        | df["text"].isna()
        | df["text"].astype(str).str.strip().eq("")
        | df["finish_reason"].astype(str).str.upper().str.contains("MAX|TOKEN", regex=True, na=False)
    )

    problems = df[row_problem_mask].copy()

    if len(df) != expected_n:
        print("\nThe number of returned rows does not match the expected size.")
        problems = df.copy()

    if len(problems):
        print(f"\nProblems found in {round_name}: {len(problems)}")
        display(problems[[
            "task_id",
            "strategy",
            "condition",
            "group_id",
            "agent_id",
            "status",
            "finish_reason",
            "usage_prompt_token_count",
            "usage_candidates_token_count",
            "usage_thoughts_token_count",
            "usage_total_token_count",
            "text",
            "error",
        ]].head(100))

        if stop_on_problem:
            raise RuntimeError(
                f"{round_name} has empty/error/MAX_TOKEN-like records. Inspect before proceeding."
            )

    print(f"\n{round_name} passed validation.")
    return problems


round1_problems = validate_parsed_round(
    df=round1_df,
    expected_n=expected_round1,
    round_name="Round 1",
    stop_on_problem=True,
)

Expected Round 1 rows: 10800
Actual Round 1 rows:   10800

Status counts:


,status,n
0,success,10800



Finish reason counts:


,finish_reason,n
0,STOP,10800



Token usage summary:


,count,mean,std,min,50%,90%,95%,99%,max
usage_prompt_token_count,10800.0,157.250000,17.341146,136.0,156.0,176.0,190.0,210.0,210.0
usage_candidates_token_count,10800.0,69.857870,81.442806,5.0,17.0,195.0,208.0,229.0,261.0
usage_thoughts_token_count,10800.0,74.630185,16.110925,32.0,72.0,92.0,102.0,135.0,325.0
usage_total_token_count,10800.0,301.738056,93.285661,185.0,250.0,445.0,463.0,494.0,545.0



Round 1 passed validation.


In [25]:
def get_task_by_id(task_id: str) -> dict:
    for task in TASK_SETTINGS:
        if task["task_id"] == task_id:
            return task
    raise KeyError(f"Unknown task_id: {task_id}")


def build_round2_plan(round1_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    r1_success = round1_df[round1_df["status"].eq("success")].copy()

    required_cols = ["task_id", "strategy", "condition", "group_id", "agent_id"]

    if r1_success.duplicated(required_cols).any():
        dupes = (
            r1_success[r1_success.duplicated(required_cols, keep=False)]
            .sort_values(required_cols)
        )
        raise ValueError(
            "Duplicate Round 1 successful records detected:\n"
            + str(dupes[required_cols + ["text"]].head(20))
        )

    for (task_id, strategy, condition, group_id), group in r1_success.groupby(
        ["task_id", "strategy", "condition", "group_id"],
        sort=True,
    ):
        task = get_task_by_id(task_id)
        group = group.sort_values("agent_index").copy()

        expected_group_size = {
            "base": 1,
            "dyad": 2,
            "triad": 3,
        }[condition]

        if len(group) != expected_group_size:
            raise ValueError(
                f"Group size mismatch for {(task_id, strategy, condition, group_id)}: "
                f"expected {expected_group_size}, got {len(group)}"
            )

        for _, ego in group.iterrows():
            peer_rows = group[group["agent_id"] != ego["agent_id"]].sort_values("agent_index")
            peer_texts = peer_rows["text"].tolist()
            peer_agent_ids = peer_rows["agent_id"].tolist()

            user_prompt = build_round2_prompt(
                task=task,
                strategy=strategy,
                condition=condition,
                self_round1=ego["text"],
                peer_round1_texts=peer_texts,
            )

            request_basis = {
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "task_set_id": TASK_SET_ID,
                "round": 2,
                "task_id": task_id,
                "task_family": task["task_family"],
                "strategy": strategy,
                "condition": condition,
                "group_id": group_id,
                "agent_id": ego["agent_id"],
                "agent_index": int(ego["agent_index"]),
                "self_round1_request_key": ego["request_key"],
                "peer_round1_agent_ids": "|".join(peer_agent_ids),
            }

            request_key = "r2__" + stable_hash(
                json.dumps(request_basis, sort_keys=True),
                24,
            )

            rows.append({
                **request_basis,
                "request_key": request_key,
                "system_instructions": SYSTEM_INSTRUCTIONS,
                "user_prompt": user_prompt,
                "temperature": TEMPERATURE,
                "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                "thinking_level": THINKING_LEVEL,
                "thinking_budget": THINKING_BUDGET,
                "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
                "self_round1_text": ego["text"],
                "peer_round1_texts_json": json.dumps(peer_texts, ensure_ascii=False),
                "created_at_utc": now_iso(),
            })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head(20)}")

    return plan_df


round2_plan_df = build_round2_plan(round1_df)

expected_round2 = expected_round1

print("Expected Round 2 requests:", expected_round2)
print("Actual Round 2 requests:  ", len(round2_plan_df))

display(
    round2_plan_df
    .groupby(["task_id", "strategy", "condition"])
    .size()
    .reset_index(name="n")
)

round2_plan_df.head()

Expected Round 2 requests: 10800
Actual Round 2 requests:   10800


,task_id,strategy,condition,n
0,aut_automobile_tire,diverge,base,150
1,aut_automobile_tire,diverge,dyad,150
2,aut_automobile_tire,diverge,triad,150
3,aut_automobile_tire,vanilla,base,150
4,aut_automobile_tire,vanilla,dyad,150
...,...,...,...,...
67,story_parachute,diverge,dyad,150
68,story_parachute,diverge,triad,150
69,story_parachute,vanilla,base,150
70,story_parachute,vanilla,dyad,150


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,system_instructions,user_prompt,temperature,max_output_tokens,thinking_level,thinking_budget,include_thinking_config_in_batch,self_round1_text,peer_round1_texts_json,created_at_utc
0,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,...,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,768,None,128,True,"A sound-absorbing, textured wall panel for a r...",[],2026-05-18T16:37:43.065318+00:00
1,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,...,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,768,None,128,True,Shredded tire crumb rubber mixed with aggregat...,[],2026-05-18T16:37:43.065837+00:00
2,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,...,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,768,None,128,True,Shredded tire rubber mixed with aggregate to c...,[],2026-05-18T16:37:43.066306+00:00
3,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,...,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,768,None,128,True,"Tire treads woven into a blast-resistant, soun...",[],2026-05-18T16:37:43.066762+00:00
4,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,...,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,768,None,128,True,Shredded tire crumb rubber mixed with aggregat...,[],2026-05-18T16:37:43.067208+00:00


In [26]:
round2_jsonl_path, round2_plan_path = make_gemini_batch_jsonl_from_plan(
    plan_df=round2_plan_df,
    round_name="round2",
    batch_input_dir=DIRS["round2_batch_inputs"],
    plan_dir=DIRS["round2_plans"],
    stem_prefix=TASK_SET_ID,
)

print("Contains thinking_config?")
print("thinking_config" in json.dumps(read_jsonl(round2_jsonl_path)[0]))

round2_batch_info = submit_gemini_batch_generic(
    batch_jsonl_path=round2_jsonl_path,
    round_name="round2",
    plan_path=round2_plan_path,
    manifest_dir=DIRS["round2_manifests"],
    uploaded_files_dir=DIRS["round2_uploaded_files"],
    display_name_prefix="deflect_creativity",
)

round2_batch_info

Wrote plan:  ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/plans/taskset_full_all_prompts_gemini25pro__round2__gemini__gemini-2.5-pro__20260518_123748__plan.csv
Wrote JSONL: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/batch_inputs/taskset_full_all_prompts_gemini25pro__round2__gemini__gemini-2.5-pro__20260518_123748__batch_input.jsonl
Requests:    10,800

First JSONL record preview:
{
  "key": "r2__7685a8b5adc449e941e8e68d",
  "request": {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "You are participating in a creativity task.\n\nObject: automobile tire\nCommon use to avoid: used on the wheel of an automobile\n\nGenerate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\nRequirements:\n- Do not use the common use.\

{'run_id': '20260518_122755__a251551c',
 'task_set_id': 'taskset_full_all_prompts_gemini25pro',
 'round': 'round2',
 'provider': 'gemini',
 'model': 'gemini-2.5-pro',
 'batch_job': {'name': 'batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj',
  'display_name': 'deflect_creativity__taskset_full_all_prompts_gemini25pro__round2__gemini-2.5-pro__20260518_122755__a251551c',
  'state': 'JOB_STATE_PENDING',
  'error': None,
  'create_time': '2026-05-18T16:37:52.993111Z',
  'start_time': None,
  'end_time': None,
  'update_time': '2026-05-18T16:37:52.993111Z',
  'model': 'models/gemini-2.5-pro',
  'src': None,
  'dest': None},
 'batch_name': 'batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj',
 'state_at_submission': 'JOB_STATE_PENDING',
 'submitted_at_utc': '2026-05-18T16:37:55.509792+00:00',
 'uploaded_file': {'name': 'files/lwj8g168sv0b',
  'display_name': 'deflect_creativity__taskset_full_all_prompts_gemini25pro__round2__gemini-2.5-pro__20260518_122755__a251551c',
  'mime_type': 'jsonl',
  'size_byt

In [30]:
round2_status = check_gemini_batch(round2_batch_info["batch_name"])

round2_status

{
  "name": "batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj",
  "display_name": "deflect_creativity__taskset_full_all_prompts_gemini25pro__round2__gemini-2.5-pro__20260518_122755__a251551c",
  "state": "JOB_STATE_SUCCEEDED",
  "error": null,
  "create_time": "2026-05-18T16:37:52.993111Z",
  "start_time": null,
  "end_time": "2026-05-18T16:40:53.018006Z",
  "update_time": "2026-05-18T16:40:53.018006Z",
  "model": "models/gemini-2.5-pro",
  "src": null,
  "dest": {
    "format": null,
    "gcs_uri": null,
    "bigquery_uri": null,
    "file_name": "files/batch-f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj",
    "inlined_responses": null,
    "inlined_embed_content_responses": null
  }
}


{'name': 'batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj',
 'display_name': 'deflect_creativity__taskset_full_all_prompts_gemini25pro__round2__gemini-2.5-pro__20260518_122755__a251551c',
 'state': 'JOB_STATE_SUCCEEDED',
 'error': None,
 'create_time': '2026-05-18T16:37:52.993111Z',
 'start_time': None,
 'end_time': '2026-05-18T16:40:53.018006Z',
 'update_time': '2026-05-18T16:40:53.018006Z',
 'model': 'models/gemini-2.5-pro',
 'src': None,
 'dest': {'format': None,
  'gcs_uri': None,
  'bigquery_uri': None,
  'file_name': 'files/batch-f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj',
  'inlined_responses': None,
  'inlined_embed_content_responses': None}}

In [31]:
round2_output_path = download_gemini_batch_results_generic(
    batch_name=round2_batch_info["batch_name"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    round_name="round2",
)

round2_output_path

Downloaded results to: ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/raw_outputs/taskset_full_all_prompts_gemini25pro__round2__a60aa59a9ab65b6d__results.jsonl


PosixPath('ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/raw_outputs/taskset_full_all_prompts_gemini25pro__round2__a60aa59a9ab65b6d__results.jsonl')

In [32]:
round2_df = parse_gemini_batch_output_to_standard_df(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    batch_name=round2_batch_info["batch_name"],
)

round2_parse_summary = save_parsed_df(
    parsed_df=round2_df,
    parsed_dir=DIRS["round2_parsed"],
    round_name="round2",
    batch_name=round2_batch_info["batch_name"],
    prefix=TASK_SET_ID,
)

print(round2_df.shape)
display(round2_df["status"].value_counts(dropna=False))
display(round2_df["finish_reason"].value_counts(dropna=False).head(20))

round2_df.head()

{
  "task_set_id": "taskset_full_all_prompts_gemini25pro",
  "round": "round2",
  "batch_name": "batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj",
  "n_records": 10800,
  "n_success": 10800,
  "n_empty_text": 0,
  "n_error": 0,
  "n_max_token_finish": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/parsed/taskset_full_all_prompts_gemini25pro__round2__gemini__gemini-2.5-pro__a60aa59a9ab65b6d__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/parsed/taskset_full_all_prompts_gemini25pro__round2__gemini__gemini-2.5-pro__a60aa59a9ab65b6d__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/02_round2/parsed/taskset_full_all_prompts_gemini25pro__round2__gemini__gemin

status
success    10800
Name: count, dtype: int64

finish_reason
STOP    10800
Name: count, dtype: int64

,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,usage_thoughts_token_count,usage_cached_content_token_count,usage_total_token_count,usage,error,batch_custom_id,batch_output_file,batch_name,raw_result_type,raw_record
0,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_key,aut,diverge,dyad,dyad_026,dyad_026__a1,...,74,None,361,"{'serviceTier': 'SERVICE_TIER_STANDARD', 'prom...",None,r2__15df73e3f1a69da344b15ad8,ai_data/deflect_creativity/gemini/model_gemini...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,response,{'response': {'responseId': 'xUALao_OF92xqtsP5...
1,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_key,aut,diverge,dyad,dyad_026,dyad_026__a2,...,54,None,337,"{'thoughtsTokenCount': 54, 'totalTokenCount': ...",None,r2__55a3dda26a935117e6ddabd0,ai_data/deflect_creativity/gemini/model_gemini...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,response,{'response': {'responseId': 'xUALatH_FsrUz7IPv...
2,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_key,aut,diverge,dyad,dyad_027,dyad_027__a1,...,119,None,400,"{'serviceTier': 'SERVICE_TIER_STANDARD', 'prom...",None,r2__01ba74bf12b31a6a0fde8ebd,ai_data/deflect_creativity/gemini/model_gemini...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,response,{'response': {'responseId': 'xUALapq2FunW_uMP4...
3,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_key,aut,diverge,dyad,dyad_027,dyad_027__a2,...,65,None,343,"{'candidatesTokenCount': 21, 'promptTokensDeta...",None,r2__5bdb865f5501cca2d6032af1,ai_data/deflect_creativity/gemini/model_gemini...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,response,{'response': {'responseId': 'xkALasKGBrihz7IPx...
4,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,2,aut_key,aut,diverge,dyad,dyad_028,dyad_028__a1,...,62,None,334,"{'serviceTier': 'SERVICE_TIER_STANDARD', 'prom...",None,r2__a9ef79340f0f57b4f7f32e08,ai_data/deflect_creativity/gemini/model_gemini...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,response,{'response': {'responseId': 'xUALaqHsHfDiqtsP0...


In [33]:
round2_problems = validate_parsed_round(
    df=round2_df,
    expected_n=expected_round2,
    round_name="Round 2",
    stop_on_problem=True,
)

Expected Round 2 rows: 10800
Actual Round 2 rows:   10800

Status counts:


,status,n
0,success,10800



Finish reason counts:


,finish_reason,n
0,STOP,10800



Token usage summary:


,count,mean,std,min,50%,90%,95%,99%,max
usage_prompt_token_count,10800.0,339.656574,197.042304,162.0,241.0,698.0,779.05,850.04,930.0
usage_candidates_token_count,10800.0,75.377037,87.282107,4.0,20.0,210.0,223.00,241.01,295.0
usage_thoughts_token_count,10800.0,71.521389,17.437133,32.0,68.0,92.0,102.00,134.00,259.0
usage_total_token_count,10800.0,486.555000,276.799389,213.0,331.0,963.0,1053.00,1157.00,1275.0



Round 2 passed validation.


In [34]:
def normalize_for_analysis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in [
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]:
        if col not in out.columns:
            out[col] = None

    keep_cols = [
        "provider",
        "model",
        "task_set_id",
        "round",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "agent_index",
        "request_key",
        "status",
        "text",
        "temperature",
        "max_output_tokens",
        "thinking_level",
        "thinking_budget",
        "include_thinking_config_in_batch",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
        "provider_response_id",
        "finish_reason",
        "usage_prompt_token_count",
        "usage_candidates_token_count",
        "usage_thoughts_token_count",
        "usage_cached_content_token_count",
        "usage_total_token_count",
        "usage",
        "error",
        "batch_name",
        "batch_custom_id",
        "batch_output_file",
        "parsed_at_utc",
    ]

    existing_keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[existing_keep_cols].copy()

    out["text_clean"] = out["text"].map(clean_model_text)
    out["is_success"] = out["status"].eq("success")

    return out


round1_analysis_df = normalize_for_analysis(round1_df)
round2_analysis_df = normalize_for_analysis(round2_df)

full_long_df = pd.concat(
    [round1_analysis_df, round2_analysis_df],
    ignore_index=True,
)

sort_cols = ["task_id", "strategy", "condition", "group_id", "agent_index", "round"]
full_long_df = full_long_df.sort_values(sort_cols).reset_index(drop=True)

print("Full long shape:", full_long_df.shape)

display(
    full_long_df
    .groupby(["round", "task_id", "strategy", "condition", "status"])
    .size()
    .reset_index(name="n")
    .head(100)
)

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

full_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.csv"
full_pkl_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.pkl"

if full_csv_path.exists() or full_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite compiled long files.")

full_long_df.to_csv(full_csv_path, index=False)
full_long_df.to_pickle(full_pkl_path)

print("Saved:")
print(full_csv_path)
print(full_pkl_path)

Full long shape: (21600, 38)


,round,task_id,strategy,condition,status,n
0,1,aut_automobile_tire,diverge,base,success,150
1,1,aut_automobile_tire,diverge,dyad,success,150
2,1,aut_automobile_tire,diverge,triad,success,150
3,1,aut_automobile_tire,vanilla,base,success,150
4,1,aut_automobile_tire,vanilla,dyad,success,150
...,...,...,...,...,...,...
95,2,aut_shoe,vanilla,triad,success,150
96,2,aut_wooden_pencil,diverge,base,success,150
97,2,aut_wooden_pencil,diverge,dyad,success,150
98,2,aut_wooden_pencil,diverge,triad,success,150


Saved:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__full_long__20260518_124349.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__full_long__20260518_124349.pkl


In [35]:
r1_small = full_long_df[full_long_df["round"].eq(1)].copy()
r2_small = full_long_df[full_long_df["round"].eq(2)].copy()

merge_keys = [
    "provider",
    "model",
    "task_set_id",
    "task_id",
    "task_family",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
    "agent_index",
]

wide_df = r1_small[merge_keys + [
    "request_key",
    "status",
    "text_clean",
    "batch_name",
    "usage_prompt_token_count",
    "usage_candidates_token_count",
    "usage_thoughts_token_count",
    "usage_total_token_count",
]].rename(
    columns={
        "request_key": "round1_request_key",
        "status": "round1_status",
        "text_clean": "round1_text",
        "batch_name": "round1_batch_name",
        "usage_prompt_token_count": "round1_usage_prompt_token_count",
        "usage_candidates_token_count": "round1_usage_candidates_token_count",
        "usage_thoughts_token_count": "round1_usage_thoughts_token_count",
        "usage_total_token_count": "round1_usage_total_token_count",
    }
).merge(
    r2_small[merge_keys + [
        "request_key",
        "status",
        "text_clean",
        "batch_name",
        "usage_prompt_token_count",
        "usage_candidates_token_count",
        "usage_thoughts_token_count",
        "usage_total_token_count",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]].rename(
        columns={
            "request_key": "round2_request_key",
            "status": "round2_status",
            "text_clean": "round2_text",
            "batch_name": "round2_batch_name",
            "usage_prompt_token_count": "round2_usage_prompt_token_count",
            "usage_candidates_token_count": "round2_usage_candidates_token_count",
            "usage_thoughts_token_count": "round2_usage_thoughts_token_count",
            "usage_total_token_count": "round2_usage_total_token_count",
        }
    ),
    on=merge_keys,
    how="outer",
    validate="one_to_one",
)

wide_df = wide_df.sort_values(
    ["task_id", "strategy", "condition", "group_id", "agent_index"]
).reset_index(drop=True)

wide_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.csv"
wide_pkl_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.pkl"

if wide_csv_path.exists() or wide_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite compiled wide files.")

wide_df.to_csv(wide_csv_path, index=False)
wide_df.to_pickle(wide_pkl_path)

print("Wide shape:", wide_df.shape)
print("Saved:")
print(wide_csv_path)
print(wide_pkl_path)

wide_df.head()

Wide shape: (10800, 30)
Saved:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__ego_wide__20260518_124349.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__ego_wide__20260518_124349.pkl


,provider,model,task_set_id,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,round2_text,round2_batch_name,round2_usage_prompt_token_count,round2_usage_candidates_token_count,round2_usage_thoughts_token_count,round2_usage_total_token_count,self_round1_request_key,peer_round1_agent_ids,self_round1_text,peer_round1_texts_json
0,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,1,...,Shredded and mixed with soil to create a perme...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,220,18,125,363,r1__439357e08e74df5ef00a8d9f,NaN,"A sound-absorbing, textured wall panel for a r...",[]
1,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,1,...,"A sound-dampening, anechoic chamber wall linin...",batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,236,27,71,334,r1__225b1c2144afb9422fa72b55,NaN,Shredded tire crumb rubber mixed with aggregat...,[]
2,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,1,...,The tire's steel bead wire is extracted and re...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,236,25,71,332,r1__78a4a8be25c3435a0a09bc6a,NaN,Shredded tire rubber mixed with aggregate to c...,[]
3,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,1,...,Finely shredded tire rubber is mixed with aggr...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,228,24,71,323,r1__5fb0ec781fffb4925a2fc285,NaN,"Tire treads woven into a blast-resistant, soun...",[]
4,gemini,gemini-2.5-pro,taskset_full_all_prompts_gemini25pro,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,1,...,The radial steel belts are extracted and woven...,batches/f8b9uy4h3bzph42qwfldy58jtzhxvdksg8sj,229,23,60,312,r1__94aacf955810843392e4ba71,NaN,Shredded tire crumb rubber mixed with aggregat...,[]


In [36]:
print("Expected long rows:", expected_round1 + expected_round2)
print("Actual long rows:  ", len(full_long_df))

print("\nExpected wide rows:", expected_round1)
print("Actual wide rows:  ", len(wide_df))

long_key_cols = [
    "provider",
    "model",
    "task_set_id",
    "round",
    "task_id",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
]

wide_key_cols = [
    "provider",
    "model",
    "task_set_id",
    "task_id",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
]

long_dupes = full_long_df.duplicated(long_key_cols).sum()
wide_dupes = wide_df.duplicated(wide_key_cols).sum()

print("\nLong duplicate keys:", long_dupes)
print("Wide duplicate keys:", wide_dupes)

display(
    full_long_df
    .groupby(["round", "task_id", "strategy", "condition", "status"])
    .size()
    .reset_index(name="n")
)

if len(full_long_df) != expected_round1 + expected_round2:
    raise RuntimeError("Compiled long row count does not match expectation.")

if len(wide_df) != expected_round1:
    raise RuntimeError("Compiled wide row count does not match expectation.")

if long_dupes > 0:
    raise RuntimeError("Duplicate long keys detected.")

if wide_dupes > 0:
    raise RuntimeError("Duplicate wide keys detected.")

print("Final structural checks passed.")

Expected long rows: 21600
Actual long rows:   21600

Expected wide rows: 10800
Actual wide rows:   10800

Long duplicate keys: 0
Wide duplicate keys: 0


,round,task_id,strategy,condition,status,n
0,1,aut_automobile_tire,diverge,base,success,150
1,1,aut_automobile_tire,diverge,dyad,success,150
2,1,aut_automobile_tire,diverge,triad,success,150
3,1,aut_automobile_tire,vanilla,base,success,150
4,1,aut_automobile_tire,vanilla,dyad,success,150
...,...,...,...,...,...,...
139,2,story_parachute,diverge,dyad,success,150
140,2,story_parachute,diverge,triad,success,150
141,2,story_parachute,vanilla,base,success,150
142,2,story_parachute,vanilla,dyad,success,150


Final structural checks passed.


In [37]:
def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\w'-]+\b", text))


def sentence_count_rough(text: str) -> int:
    if not isinstance(text, str):
        return 0
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)


validation_df = full_long_df.copy()

validation_df["word_count"] = validation_df["text_clean"].map(word_count)
validation_df["rough_sentence_count"] = validation_df["text_clean"].map(sentence_count_rough)

validation_df["slogan_word_violation"] = (
    validation_df["task_family"].eq("slogan")
    & validation_df["is_success"]
    & validation_df["word_count"].gt(6)
)

validation_df["story_sentence_flag"] = (
    validation_df["task_family"].eq("story")
    & validation_df["is_success"]
    & validation_df["rough_sentence_count"].ne(8)
)

slogan_violations = validation_df[validation_df["slogan_word_violation"]].copy()
story_sentence_flags = validation_df[validation_df["story_sentence_flag"]].copy()

print("Slogan >6-word violations:", len(slogan_violations))
display(
    slogan_violations[[
        "round",
        "task_id",
        "strategy",
        "condition",
        "agent_id",
        "text_clean",
        "word_count",
    ]].head(30)
)

print("\nStory rough sentence-count flags:", len(story_sentence_flags))
display(
    story_sentence_flags[[
        "round",
        "task_id",
        "strategy",
        "condition",
        "agent_id",
        "text_clean",
        "rough_sentence_count",
    ]].head(30)
)

token_qc_summary = (
    validation_df
    .groupby(["round", "task_family"], observed=True)
    .agg(
        n=("text_clean", "size"),
        mean_prompt_tokens=("usage_prompt_token_count", "mean"),
        mean_candidate_tokens=("usage_candidates_token_count", "mean"),
        mean_thought_tokens=("usage_thoughts_token_count", "mean"),
        mean_total_tokens=("usage_total_token_count", "mean"),
        p95_thought_tokens=("usage_thoughts_token_count", lambda x: x.quantile(0.95)),
        max_thought_tokens=("usage_thoughts_token_count", "max"),
        max_total_tokens=("usage_total_token_count", "max"),
        n_slogan_word_violations=("slogan_word_violation", "sum"),
        n_story_sentence_flags=("story_sentence_flag", "sum"),
    )
    .reset_index()
)

print("\nToken and validation QC summary:")
display(token_qc_summary)

validation_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__validation_flags__{timestamp}.csv"
token_qc_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__token_qc_summary__{timestamp}.csv"

if validation_csv_path.exists() or token_qc_csv_path.exists():
    raise FileExistsError("Refusing to overwrite validation/QC files.")

validation_df.to_csv(validation_csv_path, index=False)
token_qc_summary.to_csv(token_qc_csv_path, index=False)

print("Saved:")
print(validation_csv_path)
print(token_qc_csv_path)

Slogan >6-word violations: 73


,round,task_id,strategy,condition,agent_id,text_clean,word_count
9006,1,slogan_blood_donation,diverge,base,base_004__a1,Your blood type is the giving type.,7
9233,2,slogan_blood_donation,diverge,base,base_117__a1,Your blood type is the right type.,7
9239,2,slogan_blood_donation,diverge,base,base_120__a1,"Unite humanity, one pint at a time.",7
9312,1,slogan_blood_donation,diverge,dyad,dyad_004__a1,Your veins have a story. Share it.,7
9319,2,slogan_blood_donation,diverge,dyad,dyad_005__a2,A single pint can restart a heart.,7
9325,2,slogan_blood_donation,diverge,dyad,dyad_007__a1,Your story is waiting to be told.,7
9407,2,slogan_blood_donation,diverge,dyad,dyad_027__a2,A single pint can restart a heart.,7
9421,2,slogan_blood_donation,diverge,dyad,dyad_031__a1,A single pint can restart a heart.,7
9455,2,slogan_blood_donation,diverge,dyad,dyad_039__a2,The beat goes on because you give.,7
9513,2,slogan_blood_donation,diverge,dyad,dyad_054__a1,A single pint can restart a heart.,7



Story rough sentence-count flags: 1616


,round,task_id,strategy,condition,agent_id,text_clean,rough_sentence_count
14403,2,story_horror,diverge,base,base_002__a1,My little sister has an imaginary friend named...,7
14405,2,story_horror,diverge,base,base_003__a1,"My little sister's imaginary friend, Mr. Giggl...",9
14407,2,story_horror,diverge,base,base_004__a1,The old house my family bought came with a sin...,7
14409,2,story_horror,diverge,base,base_005__a1,"The old house came with a dumbwaiter, a relic ...",7
14412,1,story_horror,diverge,base,base_007__a1,"My little sister, Clara, insists there's a boy...",7
14419,2,story_horror,diverge,base,base_010__a1,My little brother has had an imaginary friend ...,7
14420,1,story_horror,diverge,base,base_011__a1,"My little brother, Leo, has an imaginary frien...",6
14422,1,story_horror,diverge,base,base_012__a1,The old house whispered secrets only I could h...,7
14426,1,story_horror,diverge,base,base_014__a1,"Every night, my little brother draws a picture...",7
14429,2,story_horror,diverge,base,base_015__a1,The old emergency broadcast system on my phone...,6



Token and validation QC summary:


,round,task_family,n,mean_prompt_tokens,mean_candidate_tokens,mean_thought_tokens,mean_total_tokens,p95_thought_tokens,max_thought_tokens,max_total_tokens,n_slogan_word_violations,n_story_sentence_flags
0,1,aut,4500,152.800000,16.664889,73.594667,243.059556,99.0,214,385,0,0
1,1,slogan,2700,146.000000,7.232593,77.092593,230.325185,105.0,325,487,16,0
2,1,story,3600,171.250000,183.318056,74.077778,428.645833,102.0,155,545,0,743
3,2,aut,4500,228.812444,19.032889,72.106889,319.952222,107.0,256,496,0,0
4,2,slogan,2700,203.139630,7.085185,72.108889,282.333704,102.0,259,494,57,0
5,2,story,3600,580.599444,197.026111,70.348889,847.974444,97.0,163,1275,0,873


Saved:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__validation_flags__20260518_124349.csv
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__token_qc_summary__20260518_124349.csv


In [38]:
final_manifest = {
    "run_id": RUN_ID,
    "task_set_id": TASK_SET_ID,
    "data_root": str(DATA_ROOT),
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "temperature": TEMPERATURE,
    "thinking_level": THINKING_LEVEL,
    "thinking_budget": THINKING_BUDGET,
    "gemini_batch_field_style": GEMINI_BATCH_FIELD_STYLE,
    "gemini_upload_mime_type": GEMINI_UPLOAD_MIME_TYPE,
    "include_thinking_config_in_batch": INCLUDE_THINKING_CONFIG_IN_BATCH,
    "max_output_tokens_by_family": MAX_OUTPUT_TOKENS_BY_FAMILY,
    "n_base_agents": N_BASE_AGENTS,
    "n_dyads": N_DYADS,
    "n_triads": N_TRIADS,
    "n_task_settings": len(TASK_SETTINGS),
    "task_settings": TASK_SETTINGS,
    "tiny_file_preflight_batch_info": tiny_file_preflight_batch_info,
    "tiny_file_preflight_parse_summary": tiny_file_preflight_parse_summary,
    "round1_batch_info": round1_batch_info,
    "round2_batch_info": round2_batch_info,
    "round1_parse_summary": round1_parse_summary,
    "round2_parse_summary": round2_parse_summary,
    "compiled_long_csv": str(full_csv_path),
    "compiled_long_pkl": str(full_pkl_path),
    "compiled_wide_csv": str(wide_csv_path),
    "compiled_wide_pkl": str(wide_pkl_path),
    "validation_csv": str(validation_csv_path),
    "token_qc_csv": str(token_qc_csv_path),
    "completed_at_utc": now_iso(),
}

final_manifest_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__final_manifest__{timestamp}.json"

if final_manifest_path.exists():
    raise FileExistsError(f"Refusing to overwrite final manifest: {final_manifest_path}")

write_json(final_manifest_path, final_manifest)

print("Final manifest saved:")
print(final_manifest_path)

Final manifest saved:
ai_data/deflect_creativity/gemini/model_gemini-2.5-pro/taskset_full_all_prompts_gemini25pro/run_20260518_122755__a251551c/03_compiled/taskset_full_all_prompts_gemini25pro__gemini__gemini-2.5-pro__deflect_creativity__final_manifest__20260518_124349.json
